In [1]:
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 5000)
pd.set_option('display.width', 5000)

from databases.loader import fma, musdb_oracle, musdb_demucs

## FMA-medium dataset: fully real vs fully generated

23542 files:
- stereo
- sampled at 44.1kHz
- with durations in [27.38, 30.03] sec
- with both "real" and "encodec24" versions

In [2]:
fma_df = fma()
fma_df['fpath'] = fma_df['fpath'].apply(lambda x: x[38:])

display(fma_df.head(6))

,fname,sr,dur,n_channels,fpath,codec
0,000002.mp3,44100,30.014694,2,datasets/fma_medium_encodec24/000/000002.mp3,encodec24
1,000002.mp3,44100,29.988571,2,datasets/fma_medium/000/000002.mp3,real
2,000003.mp3,44100,30.040816,2,datasets/fma_medium_encodec24/000/000003.mp3,encodec24
3,000003.mp3,44100,30.014694,2,datasets/fma_medium/000/000003.mp3,real
4,000005.mp3,44100,30.040816,2,datasets/fma_medium_encodec24/000/000005.mp3,encodec24
5,000005.mp3,44100,30.014694,2,datasets/fma_medium/000/000005.mp3,real


In [3]:
fma_stats = fma_df \
    .groupby(['fname', 'sr', 'n_channels']) \
    .agg({'codec': tuple, 'dur': 'mean'}) \
    .reset_index(drop=False) \
    .groupby(['codec', 'sr', 'n_channels']) \
    .agg({'fname': len, 'dur': [min, max]}) \

display(fma_stats)

fname        dur           
                                      len        min        max
codec             sr    n_channels                             
(encodec24, real) 44100 2           23542  27.389388  30.027755

## MUSDB18-HQ hybrid mixtures dataset

- 150 stereo mixtures at 44.1kHz with duration in [13, 628] sec
- 5 mixing gains in [-12, -6, 0, 6, 12]
- 4 mixing types ('vg_ag', 'vg_ar', 'vr_ag', 'vr_ar')
- oracle and demucs stems

In [4]:
musdb_oracle_df = musdb_oracle()
for col in ['mix_path', 'vocals_path', 'accomp_path']:
    musdb_oracle_df[col] = musdb_oracle_df[col].apply(lambda x: x[38:])

print('MUSDB ORACLE')
display(musdb_oracle_df[musdb_oracle_df['vocals_gain_db'] == 0].head(4))

MUSDB ORACLE


,track,mix_type,vocals_gain_db,sr,dur,n_channels,mix_path,vocals_path,accomp_path,codec
2,A Classic Education - NightOwl,vg_ar,0,44100,171.440181,2,datasets/musdb_hq_2stems/real_encodec24_mix/train/A Classic Education - NightOwl/vg_ar_0dB.wav,datasets/musdb_hq_2stems/encodec24/train/A Classic Education - NightOwl/vocals.wav,datasets/musdb_hq_2stems/real/train/A Classic Education - NightOwl/accompaniment.wav,encodec24
7,A Classic Education - NightOwl,vr_ag,0,44100,171.440181,2,datasets/musdb_hq_2stems/real_encodec24_mix/train/A Classic Education - NightOwl/vr_ag_0dB.wav,datasets/musdb_hq_2stems/real/train/A Classic Education - NightOwl/vocals.wav,datasets/musdb_hq_2stems/encodec24/train/A Classic Education - NightOwl/accompaniment.wav,encodec24
12,A Classic Education - NightOwl,vg_ag,0,44100,171.440181,2,datasets/musdb_hq_2stems/real_encodec24_mix/train/A Classic Education - NightOwl/vg_ag_0dB.wav,datasets/musdb_hq_2stems/encodec24/train/A Classic Education - NightOwl/vocals.wav,datasets/musdb_hq_2stems/encodec24/train/A Classic Education - NightOwl/accompaniment.wav,encodec24
17,A Classic Education - NightOwl,vr_ar,0,44100,171.440181,2,datasets/musdb_hq_2stems/real_encodec24_mix/train/A Classic Education - NightOwl/vr_ar_0dB.wav,datasets/musdb_hq_2stems/real/train/A Classic Education - NightOwl/vocals.wav,datasets/musdb_hq_2stems/real/train/A Classic Education - NightOwl/accompaniment.wav,encodec24


In [5]:
musdb_demucs_df = musdb_demucs()
for col in ['mix_path', 'vocals_path', 'accomp_path']:
    musdb_demucs_df[col] = musdb_demucs_df[col].apply(lambda x: x[38:])

print('MUSDB DEMUCS')
display(musdb_demucs_df[musdb_demucs_df['vocals_gain_db'] == 0].head(4))
# Note that only 'vocals_path' and 'accomp_path' are different between 'oracle' and 'demucs' tables

MUSDB DEMUCS


,track,mix_type,vocals_gain_db,sr,dur,n_channels,mix_path,vocals_path,accomp_path,codec
2,A Classic Education - NightOwl,vg_ar,0,44100,171.440181,2,datasets/musdb_hq_2stems/real_encodec24_mix/train/A Classic Education - NightOwl/vg_ar_0dB.wav,datasets/musdb_hq_2stems/real_encodec24_mix_demucs/train/A Classic Education - NightOwl/htdemucs/vg_ar_0dB/vocals.wav,datasets/musdb_hq_2stems/real_encodec24_mix_demucs/train/A Classic Education - NightOwl/htdemucs/vg_ar_0dB/no_vocals.wav,encodec24
7,A Classic Education - NightOwl,vr_ag,0,44100,171.440181,2,datasets/musdb_hq_2stems/real_encodec24_mix/train/A Classic Education - NightOwl/vr_ag_0dB.wav,datasets/musdb_hq_2stems/real_encodec24_mix_demucs/train/A Classic Education - NightOwl/htdemucs/vr_ag_0dB/vocals.wav,datasets/musdb_hq_2stems/real_encodec24_mix_demucs/train/A Classic Education - NightOwl/htdemucs/vr_ag_0dB/no_vocals.wav,encodec24
12,A Classic Education - NightOwl,vg_ag,0,44100,171.440181,2,datasets/musdb_hq_2stems/real_encodec24_mix/train/A Classic Education - NightOwl/vg_ag_0dB.wav,datasets/musdb_hq_2stems/real_encodec24_mix_demucs/train/A Classic Education - NightOwl/htdemucs/vg_ag_0dB/vocals.wav,datasets/musdb_hq_2stems/real_encodec24_mix_demucs/train/A Classic Education - NightOwl/htdemucs/vg_ag_0dB/no_vocals.wav,encodec24
17,A Classic Education - NightOwl,vr_ar,0,44100,171.440181,2,datasets/musdb_hq_2stems/real_encodec24_mix/train/A Classic Education - NightOwl/vr_ar_0dB.wav,datasets/musdb_hq_2stems/real_encodec24_mix_demucs/train/A Classic Education - NightOwl/htdemucs/vr_ar_0dB/vocals.wav,datasets/musdb_hq_2stems/real_encodec24_mix_demucs/train/A Classic Education - NightOwl/htdemucs/vr_ar_0dB/no_vocals.wav,encodec24


In [6]:
musdb_oracle_df \
    .groupby(['mix_type', 'vocals_gain_db', 'sr', 'n_channels']) \
    .agg({'mix_path': len, 'dur': ['mean', min, max]})


mix_path         dur                       
                                              len        mean        min         max
mix_type vocals_gain_db sr    n_channels                                            
vg_ag    -12            44100 2               150  235.730394  13.091156  628.558005
         -6             44100 2               150  235.730394  13.091156  628.558005
          0             44100 2               150  235.730394  13.091156  628.558005
          6             44100 2               150  235.730394  13.091156  628.558005
          12            44100 2               150  235.730394  13.091156  628.558005
vg_ar    -12            44100 2               150  235.730394  13.091156  628.558005
         -6             44100 2               150  235.730394  13.091156  628.558005
          0             44100 2               150  235.730394  13.091156  628.558005
          6             44100 2               150  235.730394  13.091156  628.558005
          12            44100 2               150  235.730394  13.091156  628.558005
vr_ag    -12            44100 2               150  235.730394  13.091156  628.558005
         -6             44100 2               150  235.730394  13.091156  628.558005
          0             44100 2               150  235.730394  13.091156  628.558005
          6             44100 2               150  235.730394  13.091156  628.558005
          12            44100 2               150  235.730394  13.091156  628.558005
vr_ar    -12            44100 2               150  235.730394  13.091156  628.558005
         -6             44100 2               150  235.730394  13.091156  628.558005
          0             44100 2               150  235.730394  13.091156  628.558005
          6             44100 2               150  235.730394  13.091156  628.558005
          12            44100 2               150  235.730394  13.091156  628.558005